In [1]:
#import files 

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import ast
from sklearn.neighbors import BallTree

# Locate project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED = PROJECT_ROOT / "data" / "processed"

listings_path = PROCESSED / "listings_master.parquet"
reviews_path = PROCESSED / "reviews_clean.parquet"
output_path = PROCESSED / "listings_features.parquet"

assert listings_path.exists(), f"Missing: {listings_path}"
assert reviews_path.exists(), f"Missing: {reviews_path}"

listings = pd.read_parquet(listings_path)

con = duckdb.connect()
con.execute("SET memory_limit = '12GB'")
con.execute("SET threads = 4")

print("Listings:", listings.shape)
print("Columns:", listings.columns.tolist())

Listings: (756312, 93)
Columns: ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night',

In [2]:
features = listings.copy()

# Ensure each source record has an internal unique key
features["_row_id"] = np.arange(len(features))

# Check for duplicates within locations
duplicates = features.duplicated(
    subset=["location", "id"]
).sum()

print("Duplicate location/listing IDs:", duplicates)

# Preserve original pricing and create numeric version
if "price_numeric" not in features.columns:
    features["price_numeric"] = pd.to_numeric(
        features["price"]
        .astype("string")
        .str.replace(r"[$,]", "", regex=True)
        .str.strip(),
        errors="coerce"
    )

print(
    "Valid numeric prices:",
    features["price_numeric"].notna().sum()
)

Duplicate location/listing IDs: 0
Valid numeric prices: 610695


In [3]:
# Numeric property characteristics
for col in ["accommodates", "bedrooms", "beds", "minimum_nights"]:
    if col in features.columns:
        features[f"{col}_clean"] = pd.to_numeric(
            features[col], errors="coerce"
        )

# Extract bathroom count from text such as "1.5 baths"
if "bathrooms_text" in features.columns:
    features["bathrooms_numeric"] = pd.to_numeric(
        features["bathrooms_text"]
        .astype("string")
        .str.extract(r"(\d+(?:\.\d+)?)")[0],
        errors="coerce"
    )
elif "bathrooms" in features.columns:
    features["bathrooms_numeric"] = pd.to_numeric(
        features["bathrooms"], errors="coerce"
    )

# Parse amenities safely
def parse_amenities(value):
    if isinstance(value, (list, tuple)):
        return [str(x).lower() for x in value]

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, (list, tuple)):
            return [str(x).lower() for x in parsed]
    except (ValueError, SyntaxError):
        pass

    return []

if "amenities" in features.columns:
    amenities = features["amenities"].apply(parse_amenities)

    features["amenities_count"] = amenities.apply(len)

    amenity_keywords = {
        "has_wifi": ["wifi", "wi-fi"],
        "has_air_conditioning": ["air conditioning"],
        "has_kitchen": ["kitchen"],
        "has_parking": ["parking"],
        "has_pool": ["pool"],
        "has_workspace": ["dedicated workspace"]
    }

    for new_col, keywords in amenity_keywords.items():
        features[new_col] = amenities.apply(
            lambda items: int(
                any(
                    keyword in item
                    for item in items
                    for keyword in keywords
                )
            )
        )

    del amenities

# Standardize superhost status
if "host_is_superhost" in features.columns:
    features["is_superhost"] = (
        features["host_is_superhost"]
        .astype("string")
        .str.lower()
        .map({"t": 1, "f": 0, "true": 1, "false": 0})
        .astype("Int8")
    )

# Availability rate
if "availability_365" in features.columns:
    availability = pd.to_numeric(
        features["availability_365"], errors="coerce"
    )

    features["availability_rate_365"] = (
        availability.where(availability.between(0, 365)) / 365
    )

print("Property features created.")

Property features created.


In [4]:
EARTH_RADIUS_KM = 6371.0088

# Convert coordinates to numeric
for col in ["latitude", "longitude"]:
    features[col] = pd.to_numeric(
        features[col], errors="coerce"
    )

# Identify valid coordinates
valid_geo = (
    features["latitude"].between(-90, 90)
    & features["longitude"].between(-180, 180)
    & ~(
        (features["latitude"] == 0)
        & (features["longitude"] == 0)
    )
    & features["location"].notna()
)

# Create empty features
radii = {
    "competitors_500m": 0.5,
    "competitors_1km": 1.0,
    "competitors_2km": 2.0
}

for col in radii:
    features[col] = np.nan

# Count competitors within each location
for location, group in features[valid_geo].groupby("location"):

    coords = np.radians(
        group[["latitude", "longitude"]].to_numpy()
    )

    tree = BallTree(coords, metric="haversine")

    for feature_name, radius_km in radii.items():

        counts = tree.query_radius(
            coords,
            r=radius_km / EARTH_RADIUS_KM,
            count_only=True
        ) - 1

        features.loc[group.index, feature_name] = counts

    print(f"Processed {location}: {len(group):,} listings")

Processed Amsterdam: 10,369 listings
Processed Antwerp: 2,535 listings
Processed Athens: 14,337 listings
Processed Barcelona: 15,293 listings
Processed Bergamo: 4,237 listings
Processed Berlin: 12,776 listings
Processed Bologna: 4,948 listings
Processed Bordeaux: 12,217 listings
Processed Bristol: 3,023 listings
Processed Brussels: 5,078 listings
Processed Copenhagen: 23,144 listings
Processed Crete: 27,333 listings
Processed Dublin: 6,966 listings
Processed Edinburgh: 6,244 listings
Processed Euskadi: 6,248 listings
Processed Florence: 13,472 listings
Processed Geneva: 2,593 listings
Processed Ghent: 1,383 listings
Processed Girona: 21,257 listings
Processed Greater Manchester: 6,930 listings
Processed Lisbon: 24,876 listings
Processed London: 92,638 listings
Processed Lyon: 9,317 listings
Processed Madrid: 22,708 listings
Processed Mallorca: 14,817 listings
Processed Menorca: 3,620 listings
Processed Milan: 25,750 listings
Processed Munich: 6,865 listings
Processed Naples: 11,575 lis

In [5]:
features["comparable_competitors_1km"] = np.nan

valid_comps = (
    valid_geo
    & features["room_type"].notna()
)

for (location, room_type), group in features[
    valid_comps
].groupby(["location", "room_type"]):

    coords = np.radians(
        group[["latitude", "longitude"]].to_numpy()
    )

    tree = BallTree(coords, metric="haversine")

    counts = tree.query_radius(
        coords,
        r=1.0 / EARTH_RADIUS_KM,
        count_only=True
    ) - 1

    features.loc[
        group.index, "comparable_competitors_1km"
    ] = counts

print("Comparable competition features created.")

Comparable competition features created.


In [6]:

# Connect to the reviews Parquet file
safe_path = str(reviews_path).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW reviews_clean AS
    SELECT *
    FROM read_parquet('{safe_path}')
""")

# Use latest observed review date as descriptive cutoff
cutoff = con.execute("""
    SELECT MAX(TRY_CAST(date AS DATE))
    FROM reviews_clean
""").fetchone()[0]

if cutoff is None:
    raise ValueError("No valid review dates found.")

print("Review analysis cutoff:", cutoff)

# Aggregate review counts and dates
review_features = con.execute("""
    WITH valid_reviews AS (
        SELECT
            location,
            listing_id,
            TRY_CAST(date AS DATE) AS review_date
        FROM reviews_clean
        WHERE listing_id IS NOT NULL
    )
    SELECT
        location,
        listing_id,
        COUNT(*) FILTER (
            WHERE review_date BETWEEN
                CAST(? AS DATE) - INTERVAL 29 DAYS
                AND CAST(? AS DATE)
        ) AS reviews_last_30d,

        COUNT(*) FILTER (
            WHERE review_date BETWEEN
                CAST(? AS DATE) - INTERVAL 89 DAYS
                AND CAST(? AS DATE)
        ) AS reviews_last_90d,

        COUNT(*) FILTER (
            WHERE review_date BETWEEN
                CAST(? AS DATE) - INTERVAL 364 DAYS
                AND CAST(? AS DATE)
        ) AS reviews_last_365d,

        MAX(review_date) FILTER (
            WHERE review_date <= CAST(? AS DATE)
        ) AS last_review_date

    FROM valid_reviews
    WHERE review_date <= CAST(? AS DATE)
    GROUP BY location, listing_id
""", [cutoff] * 8).df()

review_features["last_review_date"] = pd.to_datetime(
    review_features["last_review_date"]
)

review_features["days_since_last_review"] = (
    pd.Timestamp(cutoff) - review_features["last_review_date"]
).dt.days

print("Listings with review history:", len(review_features))
display(review_features.head())


Review analysis cutoff: 2026-08-11
Listings with review history: 609620


,location,listing_id,reviews_last_30d,reviews_last_90d,reviews_last_365d,last_review_date,days_since_last_review
0,Amsterdam,291752,0,4,20,2026-06-06,66
1,Amsterdam,306852,0,0,0,2023-04-01,1228
2,Amsterdam,331946,0,0,3,2025-10-26,289
3,Amsterdam,367107,0,0,3,2026-05-04,99
4,Amsterdam,503629,0,0,1,2025-08-13,363


In [7]:
features["id"] = pd.to_numeric(
    features["id"], errors="coerce"
).astype("Int64")

review_features["listing_id"] = pd.to_numeric(
    review_features["listing_id"], errors="coerce"
).astype("Int64")

features = features.merge(
    review_features.drop(columns=["last_review_date"]),
    left_on=["location", "id"],
    right_on=["location", "listing_id"],
    how="left",
    validate="many_to_one"
)

for col in [
    "reviews_last_30d",
    "reviews_last_90d",
    "reviews_last_365d"
]:
    features[col] = features[col].fillna(0).astype("int32")

features = features.drop(columns=["listing_id"])

print("Listings after review join:", len(features))

Listings after review join: 756312


In [8]:
# Choose the available neighborhood field
neighbourhood_col = next(
    (
        col for col in ["neighbourhood_cleansed", "neighbourhood"]
        if col in features.columns
    ),
    None
)

if neighbourhood_col is None:
    raise ValueError("No neighborhood column found.")

# Count listings and hosts by neighborhood
neighbourhood_summary = (
    features.dropna(subset=[neighbourhood_col])
    .groupby(
        ["country", "location", neighbourhood_col],
        dropna=False
    )
    .agg(
        neighbourhood_listing_count=("id", "size"),
        neighbourhood_host_count=("host_id", "nunique")
    )
    .reset_index()
)

features = features.merge(
    neighbourhood_summary,
    on=["country", "location", neighbourhood_col],
    how="left",
    validate="many_to_one"
)

# Total listings by location
features["location_listing_count"] = (
    features.groupby("location")["id"]
    .transform("size")
)

print("Neighborhood and market features created.")

Neighborhood and market features created.


In [12]:
# Ensure both price columns are numeric
for col in ["price_numeric", "neighbourhood_median_price"]:
    features[col] = pd.to_numeric(
        features[col],
        errors="coerce"
    )


display(
    features[
        [
            "location",
            "price_numeric",
            "neighbourhood_median_price",
            "price_vs_neighbourhood_pct"
        ]
    ].head(10)
)

# Location-level median nightly price
features["market_median_price"] = (
    features.groupby("location")["price_numeric"]
    .transform("median")
)

# Neighborhood-level median nightly price
features["neighbourhood_median_price"] = (
    features.groupby(
        ["location", neighbourhood_col]
    )["price_numeric"]
    .transform("median")
)

# Calculate relative price only for valid positive prices
valid_price = (
    (features["price_numeric"] > 0)
    & (features["neighbourhood_median_price"] > 0)
)

features["price_vs_neighbourhood_pct"] = np.nan

features.loc[
    valid_price, "price_vs_neighbourhood_pct"
] = (
    (
        features.loc[valid_price, "price_numeric"]
        / features.loc[valid_price, "neighbourhood_median_price"]
    ) - 1
) * 100

features["price_vs_neighbourhood_pct"] = (
    features["price_vs_neighbourhood_pct"].round(2)
)


print("Market price benchmarks created.")


,location,price_numeric,neighbourhood_median_price,price_vs_neighbourhood_pct
0,Amsterdam,94.0,297.52,-68.41
1,Amsterdam,<NA>,287.79,NaN
2,Amsterdam,313.67,297.52,5.43
3,Amsterdam,<NA>,287.79,NaN
4,Amsterdam,<NA>,239.8,NaN
5,Amsterdam,440.27,297.52,47.98
6,Amsterdam,574.67,287.79,99.68
7,Amsterdam,154.06,297.52,-48.22
8,Amsterdam,<NA>,286.875,NaN
9,Amsterdam,<NA>,334.0,NaN


Market price benchmarks created.


In [13]:
# Verify source row preservation
assert len(features) == len(listings), (
    "Row count changed during feature engineering."
)

# Internal row identifiers must remain unique
assert features["_row_id"].is_unique

# Ensure no impossible negative competitor counts
for col in [
    "competitors_500m",
    "competitors_1km",
    "competitors_2km",
    "comparable_competitors_1km"
]:
    assert not (features[col].dropna() < 0).any()

# Show feature completeness
feature_columns = [
    "price_numeric",
    "competitors_500m",
    "competitors_1km",
    "competitors_2km",
    "comparable_competitors_1km",
    "reviews_last_90d",
    "reviews_last_365d",
    "days_since_last_review",
    "neighbourhood_listing_count",
    "market_median_price",
    "neighbourhood_median_price",
    "price_vs_neighbourhood_pct"
]

feature_quality = pd.DataFrame({
    "missing_values": features[feature_columns].isna().sum(),
    "missing_pct": (
        features[feature_columns].isna().mean() * 100
    ).round(2)
})

display(feature_quality)

print("Final shape:", features.shape)

,missing_values,missing_pct
price_numeric,145617,19.25
competitors_500m,0,0.00
competitors_1km,0,0.00
competitors_2km,0,0.00
comparable_competitors_1km,0,0.00
reviews_last_90d,0,0.00
reviews_last_365d,0,0.00
days_since_last_review,147526,19.51
neighbourhood_listing_count,0,0.00
market_median_price,0,0.00


Final shape: (756312, 123)


In [14]:
# Remove temporary processing column
features = features.drop(columns=["_row_id"])

# Save enriched dataset
features.to_parquet(
    output_path,
    engine="pyarrow",
    compression="zstd",
    index=False
)

print("Saved:", output_path)
print("Rows:", len(features))
print("Columns:", len(features.columns))
print(
    "File size:",
    round(output_path.stat().st_size / 1024**2, 2),
    "MB"
)

Saved: C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\data\processed\listings_features.parquet
Rows: 756312
Columns: 122
File size: 305.64 MB


In [17]:
features.to_csv(
    output_path.with_suffix(".csv"),
    index=False
)